# Control Feature Visualization V3

Runnable audit harness for the v3 control feature contract.

In [ ]:
from __future__ import annotations

from pathlib import Path
import json
import sys

import pandas as pd
from IPython.display import display


def find_repo_root(start: Path | None = None) -> Path:
    start = Path.cwd() if start is None else Path(start).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "requirements.txt").exists() and (candidate / "train").exists():
            return candidate
    raise FileNotFoundError("could not find repo root")


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from train.stage1_oracle.features.control_v3 import (  # noqa: E402
    CONFIDENCE_FEATURE_NAMES,
    DEBUG_ARRAY_NAMES,
    FEATURE_NAMES,
    MODEL_FEATURE_NAMES,
    VALUE_FEATURE_NAMES,
    FeatureConfigV3,
    extract_control_features,
)
from train.stage1_oracle.features.control_v3_audit import (  # noqa: E402
    FEATURE_CONTRACT,
    feature_audit_summary,
    feature_contract_report,
    high_value_confidence_audit,
    evaluate_control_v3_audit,
    section_summaries_for_frame,
    saturation_report,
    stratified_review_queue,
)


In [ ]:
DATASET_ROOT = REPO_ROOT / "mania-dataset"
INDEX_PATH = REPO_ROOT / "train/artifacts/indexes/beatmap_index_4k_no_timing_anomalies_2to6.parquet"
AUDIT_DIR = REPO_ROOT / "train/artifacts/features/control_v3_audit"
TIMESERIES_PATH = REPO_ROOT / "train/artifacts/features/control_v3_timeseries_4k_no_timing_anomalies_2to6.parquet"
METADATA_PATH = REPO_ROOT / "train/artifacts/features/control_v3_artifact_metadata_4k_no_timing_anomalies_2to6.json"
SECTION_AUDIT_PATH = AUDIT_DIR / "control_v3_section_audit_8s_stride4.parquet"

CFG = FeatureConfigV3(grid_step=0.10)
VALUE_FEATURES = list(VALUE_FEATURE_NAMES)
CONFIDENCE_FEATURES = list(CONFIDENCE_FEATURE_NAMES)
MODEL_CHANNELS = list(MODEL_FEATURE_NAMES)
DEBUG_ARRAY_COLUMNS = list(DEBUG_ARRAY_NAMES)

if FEATURE_NAMES != MODEL_FEATURE_NAMES:
    raise RuntimeError("control_v3 FEATURE_NAMES and MODEL_FEATURE_NAMES disagree")
if "ln_change_rate_gated" not in MODEL_CHANNELS:
    raise RuntimeError("v3 model channels must include ln_change_rate_gated")
if "ln_change_rate_raw" not in DEBUG_ARRAY_COLUMNS:
    raise RuntimeError("v3 diagnostics must include ln_change_rate_raw")

print("repo root:", REPO_ROOT)
print("timeseries parquet:", TIMESERIES_PATH)
print("metadata json:", METADATA_PATH)
print("v3 model channels:", len(MODEL_CHANNELS), MODEL_CHANNELS)
print("v3 contracts:", FEATURE_CONTRACT)


In [ ]:
def build_section_audit_from_timeseries_parquet(
    *,
    timeseries_path: Path = TIMESERIES_PATH,
    section_s: float = 8.0,
    stride_s: float = 4.0,
    output_path: Path | None = SECTION_AUDIT_PATH,
) -> pd.DataFrame:
    time_df = pd.read_parquet(timeseries_path)
    records = []
    for filtered_index, group in time_df.sort_values(["filtered_index", "time_s"]).groupby("filtered_index", sort=True):
        row = group.iloc[0]
        metadata_columns = ["filtered_index", "source_index", "beatmap_id", "beatmap_set_id", "difficulty"]
        frame = group.drop(columns=[column for column in metadata_columns if column in group])
        sections = section_summaries_for_frame(row, frame, section_s=section_s, stride_s=stride_s)
        if not sections.empty:
            sections["filtered_index"] = int(filtered_index)
            for column in ["beatmap_id", "beatmap_set_id", "difficulty"]:
                if column in group:
                    sections[column] = group[column].iloc[0]
            records.append(sections)
    section_df = pd.concat(records, ignore_index=True) if records else pd.DataFrame()
    if output_path is not None:
        output_path.parent.mkdir(parents=True, exist_ok=True)
        section_df.to_parquet(output_path, index=False)
    return section_df


In [ ]:
artifact_metadata = json.loads(METADATA_PATH.read_text(encoding="utf-8")) if METADATA_PATH.exists() else {}
if artifact_metadata:
    print("artifact schema_version:", artifact_metadata.get("schema_version"))
    print("feature_contract_version:", artifact_metadata.get("feature_contract_version"))
    print("artifact map_count:", artifact_metadata.get("map_count"))
    print("artifact timeseries_rows:", artifact_metadata.get("timeseries_rows"))
    print("artifact error_count:", artifact_metadata.get("error_count"))
    print("artifact model channels:", len(artifact_metadata.get("model_feature_names", [])), artifact_metadata.get("model_feature_names", []))
else:
    print("missing metadata json:", METADATA_PATH)

if TIMESERIES_PATH.exists():
    section_df = build_section_audit_from_timeseries_parquet()
    print("section rows:", len(section_df))
    if "filtered_index" in section_df:
        print("section unique maps:", section_df["filtered_index"].nunique())
    display(feature_contract_report())
    display(saturation_report(section_df))
    display(feature_audit_summary(section_df))
    display(high_value_confidence_audit(section_df).head(25))
    display(evaluate_control_v3_audit(section_df))
    display(stratified_review_queue(section_df, n_per_bucket=25))
else:
    print("missing timeseries parquet:", TIMESERIES_PATH)
